# Завдання 2. Домашня робота

###PostgreSQL installation and librarys

In [1]:
# Install PostgreSQL server from Ubuntu packages (Colab VM runs Ubuntu)
!apt-get -qq update
!apt-get -qq install -y postgresql postgresql-contrib > /dev/null

# Start the PostgreSQL service
!service postgresql start

# Set a password for the default superuser and create a database
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres psql -c "CREATE DATABASE hw3_db;"

# SQLAlchemy 2.x + psycopg 3 driver (binary build, no compilation needed)
!pip install -q "sqlalchemy>=2.0" "psycopg[binary]"

from sqlalchemy import create_engine, text, URL
import pandas as pd

# Connection settings
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "hw3_db"

# URL.create escapes special characters in credentials safely
db_url = URL.create(
    drivername="postgresql+psycopg",  # dialect + DBAPI driver
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(db_url)

# # Open a connection, run a test query, close the connection automatically
with engine.connect() as conn:
    version = conn.execute(text("SELECT version();")).scalar_one()
    print(version)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[ OK ]
ALTER ROLE
CREATE DATABASE
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 13.3 MB/s eta 0:00:00
PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


### Download and unarchive datasets

In [2]:
!mkdir -p data/raw
!mkdir -p data/csv
!curl -L -o data/raw/seoul+bike+sharing+demand.zip https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip
!unzip data/raw/seoul+bike+sharing+demand.zip -d data/csv
# fix permissions
!chmod o+x /content /content/data /content/data/csv
!chmod o+r /content/data/csv/SeoulBikeData.csv


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  590k    0  590k    0     0  1802k      0 --:--:-- --:--:-- --:--:-- 1804k
Archive:  data/raw/seoul+bike+sharing+demand.zip
 extracting: data/csv/SeoulBikeData.csv  


### Cтворення STAGE схеми

In [3]:
# create STAGE schema
SCHEMA_SQL = text("""
DROP SCHEMA IF EXISTS hw3_stage CASCADE;
CREATE SCHEMA hw3_stage;

DROP TABLE IF EXISTS hw3_stage.raw_data, hw3_stage.bikes_data CASCADE;

CREATE TABLE hw3_stage.raw_data (
    date_rent TEXT,
    rented_bikes_count TEXT,
    hour_rent TEXT,
    temperature TEXT,
    humidity TEXT,
    wind_speed TEXT,
    visibility TEXT,
    dev_point_temperature TEXT,
    solar_radiation TEXT,
    rainfall TEXT,
    showfall TEXT,
    season TEXT,
    hollyday TEXT,
    functioning_day TEXT
);

CREATE TYPE seasons_type AS ENUM ('Autumn', 'Winter', 'Spring', 'Summer');

CREATE TABLE hw3_stage.bikes_data (
    bikes_data_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    date_rent DATE NOT NULL,
    rented_bikes_count INTEGER,
    hour_rent INTEGER NOT NULL DEFAULT 0
      CHECK(hour_rent BETWEEN 0 AND 23),
    temperature NUMERIC(4, 1),
    humidity INTEGER,
    wind_speed NUMERIC(4, 1),
    visibility INTEGER,
    dev_point_temperature NUMERIC(4, 1),
    solar_radiation NUMERIC(4, 2),
    rainfall NUMERIC(4, 1),
    showfall NUMERIC(4, 1),
    season seasons_type NOT NULL,
    hollyday boolean,
    functioning_day boolean
);
""")

with engine.begin() as conn:
    conn.execute(SCHEMA_SQL)

raw_df = pd.read_sql(text("SELECT * from hw3_stage.raw_data LIMIT 10"), engine)
raw_df
bikes_df = pd.read_sql(text("SELECT * from hw3_stage.bikes_data LIMIT 10"), engine)
bikes_df

,bikes_data_id,date_rent,rented_bikes_count,hour_rent,temperature,humidity,wind_speed,visibility,dev_point_temperature,solar_radiation,rainfall,showfall,season,hollyday,functioning_day


### Копіювання данних з  CSV у hw3_stage.raw_data

In [4]:
COPY_SQL = text("""
TRUNCATE TABLE hw3_stage.raw_data;
COPY hw3_stage.raw_data (
    date_rent,
    rented_bikes_count,
    hour_rent,
    temperature,
    humidity,
    wind_speed,
    visibility,
    dev_point_temperature,
    solar_radiation,
    rainfall,
    showfall,
    season,
    hollyday,
    functioning_day
)
FROM '/content/data/csv/SeoulBikeData.csv'
WITH (
 FORMAT csv,
 HEADER true,
 DELIMITER ',',
 ENCODING 'WIN1252'
 );""")

with engine.begin() as conn:
    conn.execute(COPY_SQL)

raw_data_df = pd.read_sql(text("SELECT * from hw3_stage.raw_data ORDER BY hour_rent DESC LIMIT 10"), engine)
raw_data_df

,date_rent,rented_bikes_count,hour_rent,temperature,humidity,wind_speed,visibility,dev_point_temperature,solar_radiation,rainfall,showfall,season,hollyday,functioning_day
0,07/12/2017,415,9,0.1,91,0.8,265,-1.1,0.14,0,0.8,Winter,No Holiday,Yes
1,09/12/2017,291,9,-4.7,48,1,1784,-14,0.23,0,0,Winter,No Holiday,Yes
2,06/12/2017,440,9,-3.3,70,0,1324,-7.9,0.25,0,0.2,Winter,No Holiday,Yes
3,02/12/2017,328,9,-2.9,68,1.6,1487,-7.9,0.3,0,0,Winter,No Holiday,Yes
4,03/12/2017,152,9,3.5,91,1.8,231,2.1,0.02,2.5,0,Winter,No Holiday,Yes
5,08/12/2017,395,9,-6.7,50,0.9,1492,-15.4,0.23,0,0,Winter,No Holiday,Yes
6,04/12/2017,509,9,-1.5,47,4.2,1888,-11.3,0.22,0,0,Winter,No Holiday,Yes
7,05/12/2017,462,9,-7.6,31,1.3,1974,-21.8,0.21,0,0,Winter,No Holiday,Yes
8,01/12/2017,490,9,-6.5,27,0.5,1928,-22.4,0.23,0,0,Winter,No Holiday,Yes
9,10/12/2017,31,9,0.2,90,1,305,-1.2,0,1.6,4.3,Winter,No Holiday,Yes


### Копіювання у типизовану таблицю hw3_stage.bikes_data

In [5]:
SQL = text("""
TRUNCATE TABLE hw3_stage.bikes_data;

WITH parsed AS (
  SELECT
    TO_DATE(NULLIF(TRIM(date_rent), ''), 'DD/MM/YYYY') AS date_rent,
    NULLIF(TRIM(rented_bikes_count), '')::INTEGER AS rented_bikes_count,
    NULLIF(TRIM(hour_rent), '')::INTEGER AS hour_rent,
    NULLIF(TRIM(temperature), '')::NUMERIC(4, 1) AS temperature,
    NULLIF(TRIM(humidity), '')::INTEGER AS humidity,
    NULLIF(TRIM(wind_speed), '')::NUMERIC(4, 1) AS wind_speed,
    NULLIF(TRIM(visibility), '')::INTEGER AS visibility,
    NULLIF(TRIM(dev_point_temperature), '')::NUMERIC(4, 1) AS dev_point_temperature,
    NULLIF(TRIM(solar_radiation), '')::NUMERIC(4, 2) AS solar_radiation,
    NULLIF(TRIM(rainfall), '')::NUMERIC(4, 1) AS rainfall,
    NULLIF(TRIM(showfall), '')::NUMERIC(4, 1) AS showfall,
    NULLIF(TRIM(season), '')::seasons_type AS season,
    CASE NULLIF(TRIM(hollyday), '')
      WHEN 'Holiday' THEN TRUE
      WHEN 'No Holiday' THEN FALSE
      ELSE NULL
    END AS hollyday,
    CASE NULLIF(TRIM(functioning_day), '')
      WHEN 'Yes' THEN TRUE
      WHEN 'No' THEN FALSE
      ELSE NULL
    END AS functioning_day
  FROM hw3_stage.raw_data
)
INSERT INTO hw3_stage.bikes_data (
    date_rent,
    rented_bikes_count,
    hour_rent,
    temperature,
    humidity,
    wind_speed,
    visibility,
    dev_point_temperature,
    solar_radiation,
    rainfall,
    showfall,
    season,
    hollyday,
    functioning_day
)
SELECT
    date_rent,
    rented_bikes_count,
    hour_rent,
    temperature,
    humidity,
    wind_speed,
    visibility,
    dev_point_temperature,
    solar_radiation,
    rainfall,
    showfall,
    season,
    hollyday,
    functioning_day
FROM parsed
WHERE date_rent IS NOT NULL and season IS NOT NULL;
""")

with engine.begin() as conn:
    conn.execute(SQL)

bikes_data_df = pd.read_sql(text("SELECT * from hw3_stage.bikes_data ORDER BY hour_rent DESC LIMIT 10"), engine)
bikes_data_df


,bikes_data_id,date_rent,rented_bikes_count,hour_rent,temperature,humidity,wind_speed,visibility,dev_point_temperature,solar_radiation,rainfall,showfall,season,hollyday,functioning_day
0,192,2017-12-08,275,23,-3.5,43,0.9,2000,-14.3,0.0,0.0,0.0,Winter,False,True
1,216,2017-12-09,265,23,-0.1,54,1.3,1570,-8.3,0.0,0.0,0.0,Winter,False,True
2,168,2017-12-07,259,23,-3.2,50,2.1,1887,-12.1,0.0,0.0,0.0,Winter,False,True
3,96,2017-12-04,248,23,-5.8,44,2.4,2000,-16.1,0.0,0.0,0.0,Winter,False,True
4,24,2017-12-01,323,23,-1.3,84,1.0,1265,-3.6,0.0,0.0,0.0,Winter,False,True
5,72,2017-12-03,303,23,3.9,90,1.3,194,2.4,0.0,0.0,0.0,Winter,False,True
6,120,2017-12-05,244,23,-2.5,59,0.0,1210,-9.4,0.0,0.0,0.0,Winter,False,True
7,144,2017-12-06,193,23,1.3,92,0.6,209,0.1,0.0,0.0,1.0,Winter,False,True
8,48,2017-12-02,366,23,4.0,56,1.0,1352,-4.0,0.0,0.0,0.0,Winter,False,True
9,240,2017-12-10,148,23,-1.4,43,4.3,2000,-12.3,0.0,0.0,0.0,Winter,False,True


## 3 Data-quality audit

### 3.1. Контроль кількості рядків

In [6]:
SQL_COUNT = text("""
SELECT 'clean' AS layer, COUNT(*) AS count FROM hw3_stage.bikes_data
UNION ALL
SELECT 'raw' AS layer, COUNT(*) AS count FROM hw3_stage.raw_data;
""")

with engine.connect() as conn:
    df = pd.read_sql(SQL_COUNT, conn)

raw_count = df.iloc[0].at['count']
clean_count = df.iloc[1].at['count']
print(f"clean: {clean_count} | raw: {raw_count}")
if raw_count == clean_count:
  print(f"Row count in clean and raw are equial.")
else:
  print("Row count in clean and raw are different.")
print("-"*100)
df

clean: 8760 | raw: 8760
Row count in clean and raw are equial.
----------------------------------------------------------------------------------------------------


,layer,count
0,clean,8760
1,raw,8760


### 3.2. Перевірка дублікатів

Для перевірці дублікатів обераємо дату і час оренди роверів - це унікальні значення.

In [7]:
SQL_DOUBLES = text("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (date_rent, hour_rent)) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (date_rent, hour_rent)) AS duplicate_rows
FROM hw3_stage.bikes_data
""")

with engine.connect() as conn:
    df = pd.read_sql(SQL_DOUBLES, conn)

df

,total_rows,distinct_business_keys,duplicate_rows
0,8760,8760,0


### 3.3. Null-rate і спеціальні маркери

In [8]:
SQL_NULL_RATE = text(
"""
SELECT
    COUNT(*) FILTER (WHERE date_rent IS NULL)::NUMERIC
    / NULLIF(COUNT(*), 0) AS date_rent_null_rate,

    COUNT(*) FILTER (WHERE rented_bikes_count IS NULL)::NUMERIC
    / NULLIF(COUNT(*), 0) AS rented_bikes_count_null_rate,

    COUNT(*) FILTER (WHERE hour_rent IS NULL)::NUMERIC
    / NULLIF(COUNT(*), 0) AS hour_rent_null_rate,

    COUNT(*) FILTER (WHERE season IS NULL)::NUMERIC
    / NULLIF(COUNT(*), 0) AS season_rate
FROM hw3_stage.bikes_data;
"""
)

with engine.connect() as conn:
    df = pd.read_sql(SQL_NULL_RATE, conn)

df

,date_rent_null_rate,rented_bikes_count_null_rate,hour_rent_null_rate,season_rate
0,0.0,0.0,0.0,0.0


## 4. Розподіли та потенційні аномалії

### 4.1. Розподіл категоріальної колонки

In [9]:
SQL_SEASONS = text(
"""
SELECT
    season,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_stage.bikes_data
GROUP BY season
ORDER BY n_rows DESC, season;
"""
)

with engine.connect() as conn:
    df = pd.read_sql(SQL_SEASONS, conn)

df

,season,n_rows,pct_rows
0,Spring,2208,25.21
1,Summer,2208,25.21
2,Autumn,2184,24.93
3,Winter,2160,24.66


**Інтерпритація:** майже однаковий розроділ по сезону, можна використовувати для навчання моделі. Данні добре збалансовані.

In [10]:
SQL_HOLLYDAYS = text(
"""
SELECT
    hollyday,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_stage.bikes_data
GROUP BY hollyday
ORDER BY n_rows DESC, hollyday;
"""
)

with engine.connect() as conn:
    df = pd.read_sql(SQL_HOLLYDAYS, conn)

df

,hollyday,n_rows,pct_rows
0,False,8328,95.07
1,True,432,4.93


**Інтерпретація:** сильно незбалансовані дані, що й очікувано. Святкових днів менше, тому й записів набагато менше. Це підтверджує, що дані для цієї ознаки цілісні.

In [11]:
SQL_FUNCTIONING = text(
"""
SELECT
    functioning_day,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_stage.bikes_data
GROUP BY functioning_day
ORDER BY n_rows DESC, functioning_day;
"""
)

with engine.connect() as conn:
    df = pd.read_sql(SQL_FUNCTIONING, conn)

df

,functioning_day,n_rows,pct_rows
0,True,8465,96.63
1,False,295,3.37


**Інтерпретація:** сильно незбалансовані дані, що й очікувано. Робочих днів набагато більш, тому й ```functioning_day = true``` записів ```~96%```. Це підтверджує, що дані для цієї ознаки цілісні.

Очикувано що у не робочі дні сервіс не працює, тому ```functioning_day = False``` можна виключити з навчання моделі.

In [12]:
SQL_FUNCTIONING_RENT = text(
"""
SELECT
    functioning_day,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows,
    ROUND(AVG(rented_bikes_count), 2) AS avg_rented_bikes_count,
    MAX(rented_bikes_count) AS max_rented_bikes_count
FROM hw3_stage.bikes_data
GROUP BY functioning_day
ORDER BY n_rows DESC, functioning_day;
"""
)

with engine.connect() as conn:
    df = pd.read_sql(SQL_FUNCTIONING_RENT, conn)

df

,functioning_day,n_rows,pct_rows,avg_rented_bikes_count,max_rented_bikes_count
0,True,8465,96.63,729.16,3556
1,False,295,3.37,0.00,0


### 4.2. Базова статистика числової колонки

In [13]:
SQL = text("""
SELECT
    COUNT(*)                                   AS n_rows,
    COUNT(rented_bikes_count)                  AS n_non_null,
    COUNT(*) - COUNT(rented_bikes_count)       AS n_null,
    MIN(rented_bikes_count)                    AS min_val,
    MAX(rented_bikes_count)                    AS max_val,
    ROUND(AVG(rented_bikes_count), 2)          AS mean_val,
    PERCENTILE_CONT(0.5)
        WITHIN GROUP (ORDER BY rented_bikes_count) AS median_val,
    ROUND(STDDEV_SAMP(rented_bikes_count), 2)  AS std_val,
    PERCENTILE_CONT(0.25)
        WITHIN GROUP (ORDER BY rented_bikes_count) AS p25,
    PERCENTILE_CONT(0.75)
        WITHIN GROUP (ORDER BY rented_bikes_count) AS p75,
    COUNT(*) FILTER (WHERE rented_bikes_count = 0) AS n_zeros
FROM hw3_stage.bikes_data;
""")
with engine.connect() as conn:
    df = pd.read_sql(SQL, conn)

df

,n_rows,n_non_null,n_null,min_val,max_val,mean_val,median_val,std_val,p25,p75,n_zeros
0,8760,8760,0,0,3556,704.6,504.5,645.0,191.0,1065.25,295


### 4.3. Один outlier-check

Перевіремо наприклад вологість та кількість орендованих роверів.

```Вологість``` повинна бути між 0 та 100 відсотків. ```humidity NOT BETWEEN 0 AND 100```

```Кількість``` роверів не може бути менша за нуль. ```rented_bikes_count < 0 ```

In [30]:
# domain check example
SQL = text("""
WITH d_check AS (
  SELECT
    -50 as min_t,
    50 as max_t,
    0 as min_h,
    100 as max_h,
    0 as min_r
)
SELECT COUNT(*) AS n_invalid
FROM hw3_stage.bikes_data d
CROSS JOIN d_check c
WHERE d.humidity NOT BETWEEN c.min_h AND c.max_h
   OR d.rented_bikes_count < c.min_r
   OR (d.temperature > c.max_t OR d.temperature < c.min_t);
""")
with engine.connect() as conn:
    df = pd.read_sql(SQL, conn)

df

,n_invalid
0,0


In [32]:
# z_score (3σ)
SQL = text("""
WITH stats AS (
    SELECT
        AVG(rented_bikes_count) AS mean_value,
        STDDEV_SAMP(rented_bikes_count) AS sd_value
    FROM hw3_stage.bikes_data
    WHERE rented_bikes_count IS NOT NULL
)
SELECT
    t.*,
    ROUND(
        ((t.rented_bikes_count - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC,
        2
    ) AS z_score
FROM hw3_stage.bikes_data AS t
CROSS JOIN stats AS s
WHERE t.rented_bikes_count IS NOT NULL
  AND s.sd_value > 0
  AND ABS(t.rented_bikes_count - s.mean_value) > 3 * s.sd_value
ORDER BY ABS(t.rented_bikes_count - s.mean_value) DESC
LIMIT 100;

""")

with engine.connect() as conn:
    df = pd.read_sql(SQL, conn)
df

,bikes_data_id,date_rent,rented_bikes_count,hour_rent,temperature,humidity,wind_speed,visibility,dev_point_temperature,solar_radiation,rainfall,showfall,season,hollyday,functioning_day,z_score
0,4819,2018-06-19,3556,18,24.1,57,2.9,1301,15.0,0.56,0.0,0.0,Summer,False,True,4.42
1,4867,2018-06-21,3418,18,27.8,43,3.0,1933,14.0,1.35,0.0,0.0,Summer,False,True,4.21
2,4651,2018-06-12,3404,18,24.9,53,3.6,2000,14.6,1.28,0.0,0.0,Summer,False,True,4.19
3,4843,2018-06-20,3384,18,27.0,55,3.1,1246,17.1,1.26,0.0,0.0,Summer,False,True,4.15
4,4459,2018-06-04,3380,18,24.4,48,1.9,1998,12.6,0.56,0.0,0.0,Summer,False,True,4.15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,3715,2018-05-04,2661,18,17.1,35,3.4,1961,1.4,1.11,0.0,0.0,Spring,False,True,3.03
63,4292,2018-05-28,2656,19,24.4,44,2.8,1979,11.3,0.45,0.0,0.0,Spring,False,True,3.03
64,4340,2018-05-30,2650,19,21.2,52,3.6,1856,10.9,0.55,0.0,0.0,Spring,False,True,3.02
65,4532,2018-06-07,2649,19,24.0,60,2.8,1379,15.7,0.38,0.0,0.0,Summer,False,True,3.01


### Інтерпритація

```Domain check``` загалом добре.

```3σ - тут немає аномалій хоча досить викидів.```

Ми бачимо сезонність та час-пік у робочі дні.

Пікові значення оренди роверів припадають на `18:00` (вечірня година пік після робочого дня) у період з травня по серпень, за комфортної температури `16-30 °C` та за відсутності опадів (дощу і снігу).

Рекомендовано додати похідну ознаку `day_of_week` (день тижня: пн-пт, сб, нд), обчислену з `date_rent`. Це дасть змогу розрізняти робочі та вихідні дні, поглибить аналіз попиту та підвищить точність прогнозу моделі.


## 5. Десять DQL/EDA-запитів

###У які години найбільше оренд?

In [38]:
SQL = text("""
SELECT
    hour_rent,
    sum(rented_bikes_count) as n_rows
FROM hw3_stage.bikes_data
WHERE hour_rent IS NOT NULL
GROUP BY hour_rent
ORDER BY n_rows DESC, hour_rent
LIMIT 10;
""")

with engine.connect() as conn:
    df = pd.read_sql(SQL, conn)
df

,hour_rent,n_rows
0,18,548568
1,19,436229
2,17,415556
3,20,390172
4,21,376479
5,8,370731
6,16,339677
7,22,336821
8,15,302653
9,14,276971


Інтерпритація

Показник часу для максимальної кількості оренди роверів у межах 17-19 годин. Це вечірня година пік після робочого дня.

###Як змінюється попит у сезони?

In [57]:
SQL = text("""
SELECT
    season,
    COUNT(DISTINCT date_rent) AS n_days,
    COUNT(DISTINCT date_rent) FILTER (WHERE hollyday) AS n_holiday_days,
    sum(rented_bikes_count) as total_rents,
    ROUND(AVG(rented_bikes_count), 1) AS avg_per_hour,
    ROUND(SUM(rented_bikes_count)::NUMERIC
          / COUNT(DISTINCT date_rent), 1) AS avg_per_day
FROM hw3_stage.bikes_data
WHERE hour_rent IS NOT NULL AND functioning_day
GROUP BY season
ORDER BY season;
""")

with engine.connect() as conn:
    df = pd.read_sql(SQL, conn)
df

,season,n_days,n_holiday_days,total_rents,avg_per_hour,avg_per_day
0,Autumn,81,4,1790002,924.1,22098.8
1,Winter,90,8,487169,225.5,5413.0
2,Spring,90,3,1611909,746.3,17910.1
3,Summer,92,2,2283234,1034.1,24817.8


Інтерпритація

Попит має виражену сезонність. Найвищий він улітку , далі йдуть осінь і весна. Взимку попит різко падає до `5 413` оренд на добу, тобто приблизно в `4.6` раза нижче, ніж улітку. Загальні обсяги можна порівнювати напряму, бо кількість днів у сезонах майже однакова (`81-92`). Основним чинником, імовірно, є погодні умови (температура, сніг). Свята на сезонний розподіл суттєво не впливають: найбільше святкових днів має зима (`8`), але це лише `~9%` днів сезону.

### Як погодні умови дозу та снігу впливають на попит

In [60]:
SQL = text("""
WITH days AS (
    SELECT
        season,
        date_rent,
        MAX(rainfall) > 0       AS has_rain,
        MAX(showfall) > 0       AS has_snow,
        SUM(rented_bikes_count) AS rents,
        COUNT(*)                AS n_hours
    FROM hw3_stage.bikes_data
    GROUP BY season, date_rent
)
SELECT
    season,
    COUNT(*) AS n_days,

    COUNT(*) FILTER (WHERE NOT has_rain AND NOT has_snow)                AS dry_days,
    COALESCE(SUM(rents) FILTER (WHERE NOT has_rain AND NOT has_snow), 0) AS rents_dry,
    ROUND(SUM(rents) FILTER (WHERE NOT has_rain AND NOT has_snow)::NUMERIC
          / SUM(n_hours) FILTER (WHERE NOT has_rain AND NOT has_snow), 1) AS avg_hourly_dry,
    ROUND(AVG(rents) FILTER (WHERE NOT has_rain AND NOT has_snow), 1)     AS avg_daily_dry,

    COUNT(*) FILTER (WHERE has_rain AND NOT has_snow)                AS rain_days,
    COALESCE(SUM(rents) FILTER (WHERE has_rain AND NOT has_snow), 0) AS rents_rain,
    ROUND(SUM(rents) FILTER (WHERE has_rain AND NOT has_snow)::NUMERIC
          / SUM(n_hours) FILTER (WHERE has_rain AND NOT has_snow), 1) AS avg_hourly_rain,
    ROUND(AVG(rents) FILTER (WHERE has_rain AND NOT has_snow), 1)     AS avg_daily_rain,

    COUNT(*) FILTER (WHERE has_snow AND NOT has_rain)                AS snow_days,
    COALESCE(SUM(rents) FILTER (WHERE has_snow AND NOT has_rain), 0) AS rents_snow,
    ROUND(SUM(rents) FILTER (WHERE has_snow AND NOT has_rain)::NUMERIC
          / SUM(n_hours) FILTER (WHERE has_snow AND NOT has_rain), 1) AS avg_hourly_snow,
    ROUND(AVG(rents) FILTER (WHERE has_snow AND NOT has_rain), 1)     AS avg_daily_snow,

    COUNT(*) FILTER (WHERE has_rain AND has_snow)                    AS rain_snow_days,
    COALESCE(SUM(rents) FILTER (WHERE has_rain AND has_snow), 0)     AS rents_rain_snow,
    ROUND(SUM(rents) FILTER (WHERE has_rain AND has_snow)::NUMERIC
          / SUM(n_hours) FILTER (WHERE has_rain AND has_snow), 1)     AS avg_hourly_rain_snow,
    ROUND(AVG(rents) FILTER (WHERE has_rain AND has_snow), 1)         AS avg_daily_rain_snow
FROM days
GROUP BY season
ORDER BY season;
""")

with engine.connect() as conn:
    df = pd.read_sql(SQL, conn)
df

,season,n_days,dry_days,rents_dry,avg_hourly_dry,avg_daily_dry,rain_days,rents_rain,avg_hourly_rain,avg_daily_rain,snow_days,rents_snow,avg_hourly_snow,avg_daily_snow,rain_snow_days,rents_rain_snow,avg_hourly_rain_snow,avg_daily_rain_snow
0,Autumn,91,68,1431762.0,877.3,21055.3,20,323389.0,673.7,16169.5,2,28374.0,591.1,14187.0,1,6477.0,269.9,6477.0
1,Winter,90,60,347585.0,241.4,5793.1,6,32146.0,223.2,5357.7,12,56380.0,195.8,4698.3,12,51058.0,177.3,4254.8
2,Spring,92,64,1295796.0,843.6,20246.8,28,316113.0,470.4,11289.8,0,0.0,NaN,NaN,0,0.0,NaN,NaN
3,Summer,92,57,1570030.0,1147.7,27544.4,35,713204.0,849.1,20377.3,0,0.0,NaN,NaN,0,0.0,NaN,NaN


Інтерпритація

Попит найвищий у сухі дні в усіх сезонах.

### Як температура впливає на попит

In [64]:
SQL = text("""
SELECT
    WIDTH_BUCKET(temperature, -30, 40, 12) AS temp_bucket,
    MIN(temperature)                       AS t_min,
    MAX(temperature)                       AS t_max,
    COUNT(*)                               AS n_hours,
    ROUND(AVG(rented_bikes_count), 0)      AS avg_per_hour
FROM hw3_stage.bikes_data
WHERE functioning_day
  AND rainfall = 0
  AND showfall = 0
GROUP BY temp_bucket
ORDER BY temp_bucket;
""")

with engine.connect() as conn:
    df = pd.read_sql(SQL, conn)
df

,temp_bucket,t_min,t_max,n_hours,avg_per_hour
0,3,-15.1,-12.6,22,169.0
1,4,-12.5,-6.7,323,171.0
2,5,-6.6,-0.9,675,223.0
3,6,-0.8,4.9,1015,340.0
4,7,5.0,10.8,1159,571.0
5,8,10.9,16.6,991,849.0
6,9,16.7,22.4,1289,1090.0
7,10,22.5,28.3,1280,1295.0
8,11,28.4,34.1,644,1177.0
9,12,34.2,39.4,136,944.0


Інтерпритація

**Попит стабільно зростає разом із температурою** (`виключення є дуже висока температура, що логічно`) - **одна з найсильніших ознак. Кореляція з орендою становить ~0.5.**

###Чим відрізняються holiday і non-holiday?

In [51]:
SQL = text("""
SELECT
    hollyday,
    COUNT(DISTINCT date_rent)             AS n_days,
    SUM(rented_bikes_count)               AS total_rented,
    ROUND(AVG(rented_bikes_count), 1)     AS avg_per_hour,
    ROUND(SUM(rented_bikes_count)::NUMERIC
          / COUNT(DISTINCT date_rent), 1) AS avg_per_day
FROM hw3_stage.bikes_data
WHERE functioning_day
GROUP BY hollyday
ORDER BY hollyday;
""")

with engine.connect() as conn:
    df = pd.read_sql(SQL, conn)
df

,hollyday,n_days,total_rented,avg_per_hour,avg_per_day
0,False,336,5956419,739.3,17727.4
1,True,17,215895,529.2,12699.7


Інтерпритація

У несвяткові дні середня погодинна оренда приблизно в 1.4 раза вища, ніж у святкові (`739` проти `529` роверів за годину, без урахування неробочих годин).

Різниця в загальних сумах (`~27` раза) зумовлена переважно кількістю днів (`336` несвяткових проти `17` святкових) і не відображає інтенсивності оренди - тільки кількість.

###Чи є нульові оренди у functional day?

In [46]:
SQL = text("""
SELECT
    functioning_day,
    sum(rented_bikes_count) as rented_bikes_count
FROM hw3_stage.bikes_data
GROUP BY functioning_day
ORDER BY rented_bikes_count DESC, functioning_day
;
""")

with engine.connect() as conn:
    df = pd.read_sql(SQL, conn)
df

,functioning_day,rented_bikes_count
0,True,6172314
1,False,0


Інтерпритація

Нульові оренди у `functioning_day` це коли севіс не працював. Можна не використовувати для навчання моделі.

## 6. Reflection

**1. Проблеми якості даних.** Пропусків і дублікатів у даних немає, але є проблеми формату. Файл збережено не в UTF-8, і символ градуса в заголовку ламав завантаження. Дати записано у форматі DD/MM/YYYY, який легко прочитати неправильно. Категоріальні поля зберігаються як текст («Holiday», «Yes»), тому їх довелося перетворювати на boolean. Нулі в години, коли сервіс не працював, - це не реальний попит, а технічний артефакт.

**2. ML-features.** Я використав би годину, температуру, опади, сезон і ознаку свята. Варто також додати день тижня, бо він сильно впливає на попит. Саму дату я не використав би напряму, як і точку роси, вологість, вітер, видимість, сонячну радіацію. Ознаку роботи сервісу краще використати для фільтрації рядків, а не як feature.

**3. Leakage.** Погодні колонки - це фактичні спостереження за годину, тож вони стають відомими лише постфактум. Для реального прогнозу їх потрібно замінити прогнозом погоди. Статус роботи сервісу теж може бути leakage, якщо його визначають після того, як година вже минула.

**4. Пропозиції для data engineer.** Варто зберігати файл в UTF-8, дати - у форматі ISO 8601, а логічні поля - як boolean. Потрібен опис колонок з одиницями виміру. Періоди простою сервісу доцільно позначати окремо, а не нулями.

**5. ML-задача.** Це regression, а точніше forecasting: прогноз погодинного попиту на велосипеди в часі.